# Human vs. Model — Spot the AI Image (all generators)

Self-contained: downloads the trained model and a set of images from this repo's GitHub
**Release**, then pits you against the model. Works in Colab or locally — no Google Drive, no
Kaggle account needed.

Images are sampled from all 8 GenImage generators; the reveal shows each fake's generator and
whether the model trained on it, so you can watch it slip on the ones it never saw.


## Config

In [1]:
import os, urllib.request, zipfile

# GitHub Release assets (public URLs — anyone can fetch these)
RELEASE = "https://github.com/LWurtzel/AI-Image-Discrimination/releases/download/v1.0"
MODEL_URL  = f"{RELEASE}/genimage_model.keras"
IMAGES_URL = f"{RELEASE}/genimage_game.zip"

MODEL_PATH  = "genimage_model.keras"
GAME_DIR    = "genimage_game"
MODEL_INPUT = 224
DISPLAY_W   = 340

TRAINED = {"stable_diffusion_v_1_4", "glide", "BigGAN"}   # for labeling only


## Download the model + images (first run only)

In [2]:
if not os.path.exists(MODEL_PATH):
    print("Downloading model (~90 MB, one time)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

if not os.path.isdir(GAME_DIR):
    print("Downloading game images...")
    urllib.request.urlretrieve(IMAGES_URL, "game.zip")
    with zipfile.ZipFile("game.zip") as z:
        z.extractall(GAME_DIR)

# be robust to how the zip nests things: find the folder holding REAL/ and FAKE/
def find_game_dir(root):
    for dp, dns, _ in os.walk(root):
        if "REAL" in dns and "FAKE" in dns:
            return dp
    return root
GAME_DIR = find_game_dir(GAME_DIR)
print("ready — images in", GAME_DIR)


HTTPError: HTTP Error 404: Not Found

## Play

In [ ]:
import glob, random
import numpy as np
from PIL import Image
import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display, clear_output

model = tf.keras.models.load_model(MODEL_PATH)

def gen_of(path):
    return os.path.basename(path).split("__")[0]

def play_game(test_dir, model, rounds=12, model_input=MODEL_INPUT, display_w=DISPLAY_W):
    reals = glob.glob(f'{test_dir}/REAL/*'); fakes = glob.glob(f'{test_dir}/FAKE/*')
    if not reals or not fakes:
        raise ValueError(f"Need both classes at {test_dir} (REAL:{len(reals)} FAKE:{len(fakes)})")
    random.shuffle(reals); random.shuffle(fakes)
    pool = [(f,'REAL') for f in reals[:rounds]] + [(f,'FAKE') for f in fakes[:rounds]]
    random.shuffle(pool)

    S = {'i':0,'human':0,'machine':0,'played':0}; name = {0:'FAKE',1:'REAL'}
    img_out, res_out = widgets.Output(), widgets.Output()
    b_real = widgets.Button(description='Real', button_style='success')
    b_fake = widgets.Button(description='Fake', button_style='danger')
    b_skip = widgets.Button(description='Skip')
    b_next = widgets.Button(description='Next ▶', button_style='info', disabled=True)
    controls = widgets.HBox([b_real, b_fake, b_skip, b_next])

    def show_image():
        with img_out:
            clear_output(wait=True)
            path,_ = pool[S['i']]
            big = Image.open(path).convert('RGB').resize((display_w, display_w), Image.LANCZOS)
            print(f"Round {S['played']+1} of {rounds}"); display(big)

    def toggle(g):
        b_real.disabled = b_fake.disabled = b_skip.disabled = not g; b_next.disabled = g

    def guess(choice):
        path, truth = pool[S['i']]
        arr = np.array(Image.open(path).convert('RGB').resize((model_input, model_input)),
                       dtype='float32')[None,...]
        prob = float(model.predict(arr, verbose=0)[0][0]); m_pred = name[int(prob>=0.5)]
        h_ok, m_ok = choice==truth, m_pred==truth
        S['human']+=h_ok; S['machine']+=m_ok; S['played']+=1
        gen = gen_of(path); tag = 'trained' if gen in TRAINED else 'UNSEEN'
        label = f"{gen} — {tag}" if truth == 'FAKE' else "real photo"
        with res_out:
            clear_output(wait=True)
            print(f"Truth:  {truth}   [{label}]")
            print(f"You:    {choice}   {'✓' if h_ok else '✗'}")
            print(f"Model:  {m_pred}  (p={prob:.2f})   {'✓' if m_ok else '✗'}")
            print(f"Score — You: {S['human']}/{S['played']}   Model: {S['machine']}/{S['played']}")
        toggle(False)

    def advance(_=None):
        S['i']+=1
        if S['played']>=rounds or S['i']>=len(pool):
            controls.close()
            with img_out: clear_output()
            with res_out:
                clear_output(wait=True)
                print(f"FINAL — You: {S['human']}/{S['played']}   Model: {S['machine']}/{S['played']}")
        else:
            toggle(True); show_image()

    b_real.on_click(lambda b: guess('REAL')); b_fake.on_click(lambda b: guess('FAKE'))
    b_skip.on_click(advance); b_next.on_click(advance)
    display(img_out, controls, res_out); show_image()

play_game(GAME_DIR, model, rounds=12)
